# Week 6 — Data Representation & Model Evaluation

**Part 1** covers representing data well (encoding, feature engineering,
missing values, outliers, leakage-safe scaling). **Part 2** covers
evaluating a model honestly once you have it (cross-validation, confusion
matrix, precision/recall, ROC/AUC) — the natural next question once your
data is in good shape.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error

## 1. Build a synthetic used-car dataset

We generate 300 used car listings with a known (but noisy) pricing rule, so we
can see clearly how much each data-representation trick helps.

In [ ]:
rng = np.random.default_rng(7)
n = 300

brands = rng.choice(["Hyundai", "Kia", "Toyota", "BMW"], size=n, p=[0.35, 0.3, 0.2, 0.15])
fuel_types = rng.choice(["gasoline", "diesel", "electric"], size=n, p=[0.55, 0.3, 0.15])
transmissions = rng.choice(["automatic", "manual"], size=n, p=[0.8, 0.2])
purchase_year = rng.integers(2010, 2024, size=n)
mileage_km = rng.integers(5_000, 180_000, size=n)

brand_premium = {"Hyundai": 0, "Kia": 0, "Toyota": 3_000, "BMW": 12_000}
fuel_premium = {"gasoline": 0, "diesel": 1_000, "electric": 6_000}

age = 2024 - purchase_year
base_price = 28_000
price = (
    base_price
    - age * 1_400                 # older car -> cheaper
    - mileage_km * 0.05           # more mileage -> cheaper
    + np.array([brand_premium[b] for b in brands])
    + np.array([fuel_premium[f] for f in fuel_types])
    + rng.normal(0, 1_500, size=n)  # noise: negotiation, condition, luck
)
price = np.clip(price, 2_000, None)

cars = pd.DataFrame({
    "brand": brands,
    "fuel_type": fuel_types,
    "transmission": transmissions,
    "purchase_year": purchase_year,
    "mileage_km": mileage_km,
    "price": price.round(0),
})
cars.head()

## Step 1: Numbers-only baseline (ignore the categories)

The naive approach: drop every text column and use only the numeric ones.

In [ ]:
def evaluate(X, y, label):
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=42)
    model = LinearRegression().fit(Xtr, ytr)
    mae = mean_absolute_error(yte, model.predict(Xte))
    print(f"{label:35s} MAE = ${mae:,.0f}")
    return mae

results = {}
X1 = cars[["purchase_year", "mileage_km"]]
results["1. numeric only"] = evaluate(X1, cars["price"], "1. Numeric columns only")

## Step 2: One-hot encode the categorical columns

`pd.get_dummies` turns each category into its own 0/1 column, so "brand" (a
word) becomes 4 numeric columns the model can use — without implying any order
between brands.

In [ ]:
categorical_encoded = pd.get_dummies(cars[["brand", "fuel_type", "transmission"]])
categorical_encoded.head()

In [ ]:
X2 = pd.concat([cars[["purchase_year", "mileage_km"]], categorical_encoded], axis=1)
results["2. + one-hot categories"] = evaluate(X2, cars["price"], "2. + one-hot encoded categories")

## Step 3: Feature engineering — `age` instead of `purchase_year`

A linear model can only learn "the *bigger* this number, the *more/less* the
price changes." `purchase_year` (e.g. 2010 vs. 2023) works, but `age` (car's
age in years) is a more natural, directly-meaningful feature for the same
underlying relationship. Feature engineering means *reshaping* raw values into
a form that's easier for the model to use.

In [ ]:
cars["age"] = 2024 - cars["purchase_year"]

X3 = pd.concat([cars[["age", "mileage_km"]], categorical_encoded], axis=1)
results["3. + engineered 'age' feature"] = evaluate(X3, cars["price"], "3. + engineered 'age' feature")

## Step 4: Scale the numeric features

`mileage_km` ranges up to 180,000 while `age` maxes out around 14 — very
different scales. Some models (and especially neural networks, coming in
Week 8) train better when every feature is standardized to a similar range.

In [ ]:
numeric_cols = ["age", "mileage_km"]
scaler = StandardScaler()
X4 = X3.copy()
X4[numeric_cols] = scaler.fit_transform(X4[numeric_cols])

results["4. + scaled numeric features"] = evaluate(X4, cars["price"], "4. + scaled numeric features")

## Compare all four representations

In [ ]:
summary = pd.Series(results)
print(summary)

plt.figure(figsize=(7, 4))
summary.plot(kind="barh", color="steelblue")
plt.title("Prediction Error (MAE, lower = better) by Data Representation")
plt.xlabel("Mean Absolute Error ($)")
plt.gca().invert_yaxis()
plt.show()

## 5. Real-world practice: Titanic — missing values, outliers, leakage-safe scaling

So far we've used a synthetic used-car dataset to practice encoding and
feature engineering. Now let's use real
[Titanic](https://www.kaggle.com/datasets/yasserh/titanic-dataset) passenger
data to practice the rest of this week's preprocessing topics — **missing
values**, **outliers**, and **avoiding data leakage when scaling**. This is a
plain dataset (not a Kaggle competition), so it downloads with no login
required.

In [ ]:
!pip install -q kagglehub
import kagglehub
import glob

titanic_dir = kagglehub.dataset_download("yasserh/titanic-dataset")
csv_path = glob.glob(f"{titanic_dir}/**/*.csv", recursive=True)[0]
print("Using file:", csv_path)

titanic = pd.read_csv(csv_path)
print("Shape:", titanic.shape)
titanic.head()

### Checking for missing values

As covered in lecture, missing values don't mean we should just drop every
row that has one — real-world data has missing values all the time, and
dropping too aggressively throws away good data. First let's see which
columns have how much missing.

In [ ]:
print(titanic.isna().sum())
print(f"\nAge missing: {titanic['Age'].isna().mean():.1%}")
print(f"Cabin missing: {titanic['Cabin'].isna().mean():.1%}")
print(f"Embarked missing: {titanic['Embarked'].isna().mean():.1%}")

### Three different strategies for three different columns

Matching the strategies from lecture to what each column actually needs:

- **`Cabin`**: over 70% missing — at this rate, imputing is unreliable, so
  we **drop the whole column** instead.
- **`Embarked`**: only 2 missing values — fill with the **mode** (most
  common value).
- **`Age`**: about 20% missing — too much to drop, but a single global mean
  would be crude. We use the lecture's "mean within the same category" idea:
  group by `Pclass` and `Sex`, and fill with **that group's median** (median
  is less sensitive to extreme values than mean).

In [ ]:
titanic = titanic.drop(columns=["Cabin"])
print("Dropped Cabin (over 70% missing)")

embarked_mode = titanic["Embarked"].mode()[0]
titanic["Embarked"] = titanic["Embarked"].fillna(embarked_mode)
print(f"Filled Embarked missing values with the mode: '{embarked_mode}'")

titanic["Age"] = titanic.groupby(["Pclass", "Sex"])["Age"].transform(lambda s: s.fillna(s.median()))
titanic["Age"] = titanic["Age"].fillna(titanic["Age"].median())  # fallback if a group has no valid values at all
print("Filled Age missing values with the (Pclass, Sex) group median")

print("\nMissing values after cleaning:")
print(titanic.isna().sum())

### A deeper look: does the imputation method actually change the result?

The three fixes above were deliberately simple — `Age` only really correlates
with `Pclass`/`Sex`, so a group median already captures most of what a
fancier method could add, and `Cabin` is too sparse to impute at all. But
lecture also introduced more powerful options: filling with a column's
**mean**, **median**, or **mode**, a **regression model** that predicts the
missing value from other columns, and **multiple imputation (MICE)**, which
fills every missing cell by iteratively regressing each column on all the
others. Titanic's `Age` just isn't a good dataset to tell these apart.

A better showcase: the **user x movie rating matrix** from Week 5's movie
clustering capstone — lecture's own missing-value slide actually used a tiny
version of exactly this (a handful of people x a few movies, with blanks).
Real [MovieLens](https://movielens.org/) ratings have plenty of genuinely
correlated columns (liking one action movie predicts liking another), so the
five methods actually have a chance to disagree — and we can check,
concretely, whether that disagreement changes anything downstream.

In [ ]:
import urllib.request

MOVIELENS_BASE_URL = "https://raw.githubusercontent.com/digantamisra98/KMeans-Movie-Ratings/master"
urllib.request.urlretrieve(f"{MOVIELENS_BASE_URL}/movies.csv", "ml_movies.csv")
urllib.request.urlretrieve(f"{MOVIELENS_BASE_URL}/ratings.csv", "ml_ratings.csv")

ml_ratings = pd.read_csv("ml_ratings.csv")
ml_movies = pd.read_csv("ml_movies.csv")
print("ratings:", ml_ratings.shape, " movies:", ml_movies.shape)
print("users:", ml_ratings["userId"].nunique(), " movies rated:", ml_ratings["movieId"].nunique())

### Rebuilding Week 5's user x reference-movie matrix — but keeping the blanks this time

Same idea as Week 5: pick a genre-balanced set of popular "reference
movies," and represent each eligible user as their rating vector over just
those movies. The difference is we **don't fill anything in yet** — we want
the real blanks to actually compare imputation methods against.

In [ ]:
genre_targets = {
    "Action": 5, "Comedy": 5, "Drama": 5, "Sci-Fi": 5,
    "Animation": 3, "Romance": 3, "Horror": 2,
}
POPULAR_MIN = 50

rating_counts = ml_ratings.groupby("movieId").size().rename("n_ratings")
ml_movies_stats = ml_movies.merge(rating_counts, on="movieId")
ml_popular = ml_movies_stats[ml_movies_stats["n_ratings"] >= POPULAR_MIN]

reference_ids = []
for genre, n in genre_targets.items():
    candidates = ml_popular[ml_popular["genres"].str.contains(genre)].sort_values("n_ratings", ascending=False)
    reference_ids.extend(candidates["movieId"].head(n).tolist())
reference_ids = list(dict.fromkeys(reference_ids))
reference_movies = ml_movies_stats[ml_movies_stats["movieId"].isin(reference_ids)].reset_index(drop=True)

ELIGIBLE_FRAC = 0.5
ref_ratings = ml_ratings[ml_ratings["movieId"].isin(reference_movies["movieId"])]
n_ref = len(reference_movies)
user_ref_counts = ref_ratings.groupby("userId")["movieId"].nunique()
eligible_users = user_ref_counts[user_ref_counts >= ELIGIBLE_FRAC * n_ref].index

movie_pivot = ref_ratings[ref_ratings["userId"].isin(eligible_users)].pivot_table(
    index="userId", columns="movieId", values="rating"
)
movie_pivot = movie_pivot.reindex(columns=reference_movies["movieId"].values)

n_missing = movie_pivot.isna().sum().sum()
n_observed = movie_pivot.notna().sum().sum()
print(f"matrix shape: {movie_pivot.shape}  (users x reference movies)")
print(f"observed ratings: {n_observed}   missing: {n_missing}  ({n_missing / (n_missing + n_observed):.1%})")

### Setting up a fair test: hide known ratings, then see which method guesses them best

We can't check a method's guess against a rating nobody ever gave. So
instead, we take ratings we **do** know, hide 20% of them at random, and see
how close each method's fill comes to the real value — the smaller the
error, the better that method is at recovering a genuinely missing rating.

In [ ]:
rng = np.random.default_rng(42)
observed_positions = np.argwhere(movie_pivot.notna().values)
n_hide = int(0.2 * len(observed_positions))
hidden = observed_positions[rng.choice(len(observed_positions), size=n_hide, replace=False)]

true_values = movie_pivot.values[hidden[:, 0], hidden[:, 1]].copy()
masked_values = movie_pivot.values.copy()
masked_values[hidden[:, 0], hidden[:, 1]] = np.nan
masked_pivot = pd.DataFrame(masked_values, index=movie_pivot.index, columns=movie_pivot.columns)

print(f"hid {n_hide} known ratings to use as a held-out answer key")

### Five imputation methods

- **Mean** — fill with that movie's average rating across all other users.
- **Median** — fill with that movie's median rating.
- **Mode** — fill with that movie's single most common rating value.
- **Regression** — predict a user's missing rating for a movie from their
  ratings on the *other* movies, using a `LinearRegression` trained on users
  who rated it.
- **Multiple imputation (MICE)** — scikit-learn's `IterativeImputer`:
  repeatedly regress each column on all the others, letting every column's
  estimate refine every other column's.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.experimental import enable_iterative_imputer  # noqa: F401 -- unlocks IterativeImputer
from sklearn.impute import IterativeImputer
from sklearn.metrics import mean_squared_error

def impute_all_methods(df):
    filled = {
        "mean": df.apply(lambda col: col.fillna(col.mean()), axis=0).values,
        "median": df.apply(lambda col: col.fillna(col.median()), axis=0).values,
        "mode": df.apply(lambda col: col.fillna(col.mode().iloc[0]), axis=0).values,
    }

    # Single-pass regression: use the mean-filled matrix for complete predictor
    # columns, then predict each movie's missing ratings from the rest.
    predictors = filled["mean"]
    regression_filled = df.values.copy()
    for j in range(df.shape[1]):
        column = df.values[:, j]
        is_missing = np.isnan(column)
        if not is_missing.any():
            continue
        other_cols = [k for k in range(df.shape[1]) if k != j]
        model = LinearRegression().fit(predictors[~is_missing][:, other_cols], column[~is_missing])
        regression_filled[is_missing, j] = model.predict(predictors[is_missing][:, other_cols])
    filled["regression"] = regression_filled

    filled["MICE"] = IterativeImputer(random_state=0, max_iter=50, tol=1e-2).fit_transform(df.values)
    return filled

def held_out_rmse(filled_values):
    predicted = filled_values[hidden[:, 0], hidden[:, 1]]
    return np.sqrt(mean_squared_error(true_values, predicted))

masked_fills = impute_all_methods(masked_pivot)
rmse_by_method = pd.Series({name: held_out_rmse(f) for name, f in masked_fills.items()}).sort_values()
print(rmse_by_method)

plt.figure(figsize=(7, 4))
rmse_by_method.plot(kind="barh", color="steelblue")
plt.title("Imputation Accuracy on Held-Out Ratings (RMSE, lower = better)")
plt.xlabel("RMSE")
plt.gca().invert_yaxis()
plt.show()

**Mode comes out clearly worst** — collapsing every blank to a single most
common rating throws away the fact that ratings aren't really categorical.
**Regression tends to do best** here — it has genuinely correlated columns
to lean on (someone's rating on one action movie predicts another).
**MICE lands in the same range as mean/median** on this particular matrix —
with this few, often weakly-correlated columns, there isn't always enough
shared structure for its iterative refinement to beat a flat average. That's
a useful lesson on its own: a more sophisticated-sounding method isn't
automatically better — it's worth actually measuring, the way we just did,
rather than assuming.

### Does this change the final recommendation?

A lower RMSE is convincing on its own, but let's connect it to something
concrete: if we fill in the **real** (not held-out) matrix with each method
and re-run Week 5's exact clustering step, do any users end up in a
different taste cluster — and does that change their recommended movies?

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score

full_fills = impute_all_methods(movie_pivot)

N_CLUSTERS = 4
cluster_labels = {}
for name, filled in full_fills.items():
    km = KMeans(n_clusters=N_CLUSTERS, n_init=10, random_state=42)
    cluster_labels[name] = pd.Series(km.fit_predict(filled), index=movie_pivot.index)
cluster_labels_df = pd.DataFrame(cluster_labels)

print("Agreement with the 'mean' clustering (Adjusted Rand Index, 1.0 = identical):")
for name in ["median", "mode", "regression", "MICE"]:
    ari = adjusted_rand_score(cluster_labels_df["mean"], cluster_labels_df[name])
    print(f"  {name:12s} {ari:.3f}")

An Adjusted Rand Index well below 1.0 means a real share of users get
sorted into a *different* taste group purely because of which imputation
method filled their blanks — not because their actual ratings changed.
Let's see it play out for one specific user.

In [ ]:
MIN_CLUSTER_RATINGS = 3

label_comparison = cluster_labels_df.copy()
label_comparison["n_missing"] = movie_pivot.isna().sum(axis=1)
fully_disagreeing = label_comparison[
    (label_comparison["mean"] != label_comparison["regression"])
    & (label_comparison["regression"] != label_comparison["MICE"])
    & (label_comparison["mean"] != label_comparison["MICE"])
]
example_user = fully_disagreeing.sort_values("n_missing", ascending=False).index[0]
n_rated = len(reference_movies) - label_comparison.loc[example_user, "n_missing"]
print(f"User {example_user} rated only {n_rated:.0f} of the {len(reference_movies)} reference movies, "
      f"and lands in a different cluster under every one of mean / regression / MICE: "
      f"{label_comparison.loc[example_user, ['mean', 'regression', 'MICE']].to_dict()}")

for method in ["mean", "regression", "MICE"]:
    labels = cluster_labels[method]
    ratings_labeled = ml_ratings[ml_ratings["userId"].isin(eligible_users)].merge(
        labels.rename("cluster"), left_on="userId", right_index=True
    )
    cluster_movie_stats = (
        ratings_labeled.groupby(["cluster", "movieId"])["rating"]
        .agg(["mean", "count"])
        .reset_index()
    )
    cluster_movie_stats = cluster_movie_stats[cluster_movie_stats["count"] >= MIN_CLUSTER_RATINGS]
    cluster_movie_stats = cluster_movie_stats[~cluster_movie_stats["movieId"].isin(reference_movies["movieId"])]

    user_cluster = labels.loc[example_user]
    top5 = (
        cluster_movie_stats[cluster_movie_stats["cluster"] == user_cluster]
        .merge(ml_movies, on="movieId")
        .sort_values("mean", ascending=False)
        .head(5)
    )
    print(f"\n--- filled with '{method}' -> user {example_user} assigned to cluster {user_cluster} ---")
    print(top5[["title", "mean", "count"]].to_string(index=False))

**Same user, same real opinions — three different recommendation lists,**
purely because of which missing-value method filled in the blanks before
clustering. This is the concrete version of today's lecture point:
missing-value handling isn't a mechanical step to get past quickly, it's a
modeling decision with real downstream consequences, and it's worth
validating empirically (like the held-out RMSE test above) rather than
picking a method out of habit.

Back to the Titanic pipeline — next we look at outliers.

### Finding outliers

Plotting `Fare` as a boxplot shows most passengers paid a modest fare, while
a handful paid a lot more. Of the methods mentioned in lecture (IQR,
Z-score, LOF, DBSCAN), we'll use the simplest: the **IQR method** — anything
outside `Q1 - 1.5*IQR` to `Q3 + 1.5*IQR` counts as an outlier.

In [ ]:
plt.figure(figsize=(6, 4))
plt.boxplot(titanic["Fare"])
plt.title("Fare Distribution (boxplot)")
plt.ylabel("Fare")
plt.show()

Q1, Q3 = titanic["Fare"].quantile([0.25, 0.75])
IQR = Q3 - Q1
lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
outliers = titanic[(titanic["Fare"] < lower) | (titanic["Fare"] > upper)]

print(f"IQR range: [{lower:.2f}, {upper:.2f}]")
print(f"Passengers flagged as outliers: {len(outliers)} / {len(titanic)}")
outliers[["Pclass", "Sex", "Age", "Fare"]].sort_values("Fare", ascending=False).head(10)

**Remove them, or keep them?** As lecture emphasized, an outlier isn't
automatically an error. Here, the high-fare passengers are mostly genuine
**1st-class (Pclass=1) passengers** — not a measurement mistake, just people
who paid for an expensive ticket. In cases like this, it's common to either
**leave them as-is**, or, if a model turns out to be overly sensitive to
extreme values, **cap** them at some threshold (winsorizing) instead of
deleting rows outright.

### Encoding categorical columns: ordinal vs. nominal

Applying the distinction from lecture to this dataset:

- **`Pclass`** (1st < 2nd < 3rd class) is an **ordinal** category — it's
  already numeric, and that order is meaningful, so we can **use it as-is**.
- **`Sex`** and **`Embarked`** are **nominal** categories with no order, so
  we use **one-hot encoding** like in the earlier section.

In [ ]:
cat_encoded = pd.get_dummies(titanic[["Sex", "Embarked"]])
numeric_cols = ["Age", "Fare", "SibSp", "Parch"]

X = pd.concat([titanic[["Pclass"] + numeric_cols], cat_encoded], axis=1)
y = titanic["Survived"]
X.head()

### Splitting into train / validation / test, without leaking into the scaler

Two rules from lecture, applied here:

1. **Three-way split**: divide the data into train/validation/test (a common
   ratio is 6:2:2 or 7:2:1). Validation is used during training to pick the
   best model and tune hyperparameters; test is touched exactly once, at the
   very end.
2. **Scale using training data only**: when you `fit` a `StandardScaler`, it
   must only see the **training** data. Fitting it on the full dataset (or
   anything that includes val/test) leaks a bit of "future" information into
   the mean/std it learns — that's **data leakage**.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=0.25, random_state=42, stratify=y_trainval
)
print(f"train/val/test sizes: {len(X_train)} / {len(X_val)} / {len(X_test)}")
print(f"ratios: {len(X_train)/len(X):.2f} / {len(X_val)/len(X):.2f} / {len(X_test)/len(X):.2f}")

In [ ]:
# Wrong way: fit the scaler on the full dataset (val/test information leaks in)
scaler_wrong = StandardScaler().fit(X[numeric_cols])

# Right way: fit only on train, then just transform() val/test
scaler_right = StandardScaler().fit(X_train[numeric_cols])

print(f"Age mean/std, fit on FULL data (leaky):   {scaler_wrong.mean_[0]:.3f} / {scaler_wrong.scale_[0]:.3f}")
print(f"Age mean/std, fit on TRAIN only (correct): {scaler_right.mean_[0]:.3f} / {scaler_right.scale_[0]:.3f}")

X_train_scaled = X_train.copy()
X_val_scaled = X_val.copy()
X_test_scaled = X_test.copy()
X_train_scaled[numeric_cols] = scaler_right.transform(X_train[numeric_cols])
X_val_scaled[numeric_cols] = scaler_right.transform(X_val[numeric_cols])
X_test_scaled[numeric_cols] = scaler_right.transform(X_test[numeric_cols])

print("\ntrain mean after scaling (should be ~0):", X_train_scaled[numeric_cols].mean().mean().round(6))

Notice the two mean/std values are (slightly) different. This dataset is
small enough that the difference is minor, but with a larger dataset, or
train/val/test sets with different distributions, that gap can actually
affect model performance — and more fundamentally, it breaks the premise
that val/test are supposed to look like "data the model hasn't seen yet."

**To recap:** handling missing values appropriately per column, flagging
outliers and deciding whether to keep or cap them, encoding ordinal vs.
nominal categories correctly, and fitting the scaler on training data
only — all of this is the **Data Preprocessing** stage from the ML
pipeline at the start of lecture. **Part 2**, right below, uses these same validation/test splits to
actually compute metrics like the confusion matrix, precision, and
recall.

## Part 2: Model Evaluation — Breast Cancer Diagnosis Evaluator

**Theme:** Model evaluation — cross-validation & evaluation metrics

So far we've measured models with a single train/test split and a single
accuracy number. That's not enough for a real decision — especially in a
medical context, where the *kind* of mistake matters a lot: missing a
malignant tumor (false negative) is far worse than an unnecessary follow-up
test (false positive).

**Dataset:** scikit-learn's built-in breast cancer dataset — 569 patients,
30 measurements from a tumor scan, label = malignant or benign.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import (
    accuracy_score, confusion_matrix, ConfusionMatrixDisplay,
    classification_report, roc_curve, roc_auc_score, precision_recall_curve,
)

In [ ]:
data = load_breast_cancer()
X, y = data.data, data.target
print("Classes:", dict(zip(data.target_names, [0, 1])))
print("Shape:", X.shape)
print("Class balance:", np.bincount(y), "(0=malignant, 1=benign)")

## 1. A single train/test split — and why it's not enough

We build a `Pipeline` so scaling always happens *inside* the same fit/predict
step (never fit the scaler on test data — that would leak information).

In [ ]:
model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000))

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=1, stratify=y
)
model.fit(X_train, y_train)
single_split_acc = accuracy_score(y_test, model.predict(X_test))
print(f"Accuracy on THIS ONE split: {single_split_acc:.2%}")

In [ ]:
# What if we'd used a different random split? Try several seeds.
accs = []
for seed in range(10):
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=seed, stratify=y)
    m = make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000)).fit(Xtr, ytr)
    accs.append(accuracy_score(yte, m.predict(Xte)))

print("Accuracy across 10 different random splits:", [f"{a:.2%}" for a in accs])
print(f"Range: {min(accs):.2%} to {max(accs):.2%} — a single split can be lucky or unlucky!")

## 2. k-fold cross-validation: a more honest estimate

Instead of one split, **k-fold cross-validation** splits the data into `k`
parts, trains on `k-1` of them and tests on the last, and rotates — so every
example gets used for testing exactly once. We report the average.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(model, X, y, cv=cv, scoring="accuracy")

print("5-fold CV accuracy per fold:", [f"{s:.2%}" for s in cv_scores])
print(f"Mean: {cv_scores.mean():.2%}  (+/- {cv_scores.std():.2%})")

## 3. Confusion matrix: what kind of mistakes does it make?

Accuracy hides *which* mistakes happen. The confusion matrix doesn't.

In [ ]:
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=data.target_names)
disp.plot(cmap="Blues")
plt.title("Confusion Matrix")
plt.show()

print(classification_report(y_test, y_pred, target_names=data.target_names))

## 4. Precision, recall, and why accuracy alone can mislead

- **Precision** — of everything we predicted "benign", how much really was benign?
- **Recall** — of everything that really was "malignant", how much did we catch?

In this task, missing a malignant tumor (a false negative for the malignant
class) is the costly mistake — so **recall on the malignant class** matters a
lot, sometimes more than overall accuracy.

In [ ]:
# malignant is class 0 in this dataset — let's look at recall for it specifically
report = classification_report(y_test, y_pred, target_names=data.target_names, output_dict=True)
print(f"Recall on malignant cases: {report['malignant']['recall']:.2%}")
print(f"Precision on malignant cases: {report['malignant']['precision']:.2%}")

## 5. ROC curve and AUC

The model doesn't just output a class — it outputs a *probability*. The ROC
curve shows the tradeoff between catching more true positives (recall) and
generating more false alarms, as we sweep the decision threshold.

In [ ]:
y_scores = model.predict_proba(X_test)[:, 1]  # probability of "benign"
fpr, tpr, thresholds = roc_curve(y_test, y_scores)
auc = roc_auc_score(y_test, y_scores)

plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, label=f"Logistic Regression (AUC = {auc:.3f})")
plt.plot([0, 1], [0, 1], "k--", label="random guessing")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate (Recall)")
plt.title("ROC Curve")
plt.legend()
plt.show()